# 01 — EDA & Baselines (KNN / Logistic Regression)

Goals:
- Sanity‑check the data (shapes, nulls, class balance).
- Visualize a few sample digits.
- Train quick baselines (KNN & Logistic Regression) with a small validation split.
- Write a Kaggle submission (`submissions/submission_baseline.csv`) from the better baseline.

In [ ]:
# Minimal deps (Kaggle already has these; the line below is here for portability)
%pip -q install scikit-learn --disable-pip-version-check

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from src.utils import seed_everything, load_train_test, as_images, plot_samples, save_submission

seed_everything(42)
DATA_DIR = "data"
SUBMISSION_PATH = "submissions/submission_baseline.csv"

os.makedirs("submissions", exist_ok=True)

## 1) Load & sanity checks

In [ ]:
# Load train/test (pixels normalized to [0,1])
X, y, X_test = load_train_test(DATA_DIR)
print("Train shape:", X.shape, "Labels:", y.shape)
print("Test shape:", X_test.shape)
print("Missing values in train:", np.isnan(X).sum() + np.isnan(y).sum())
print("Missing values in test:", np.isnan(X_test).sum())

In [ ]:
# Class distribution
vals, counts = np.unique(y, return_counts=True)
plt.figure(figsize=(6,3))
plt.bar(vals, counts)
plt.title("Class distribution")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

In [ ]:
# Peek at some samples
plot_samples(X, y, n=16, title="Training samples")

## 2) Validation split

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
Xtr.shape, Xva.shape

## 3) Baseline A — KNN (k=3, distance weights)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1)
knn.fit(Xtr, ytr)
knn_acc = accuracy_score(yva, knn.predict(Xva))
print("KNN validation accuracy:", knn_acc)

## 4) Baseline B — Multinomial Logistic Regression

In [ ]:
# Logistic Regression tends to like scaled inputs (we already have [0,1]).
# Increase max_iter to ensure convergence.
logreg = LogisticRegression(max_iter=300, multi_class="multinomial", solver="lbfgs", n_jobs=-1)
logreg.fit(Xtr, ytr)
lr_acc = accuracy_score(yva, logreg.predict(Xva))
print("LogReg validation accuracy:", lr_acc)

## 5) Pick best baseline and train on full train

In [ ]:
best_name, best_model = ("KNN", knn) if knn_acc >= lr_acc else ("LogReg", logreg)
print(f"Best baseline: {best_name} (val acc = {max(knn_acc, lr_acc):.4f})")

# Refit on full training set for final submission
best_model.fit(X, y)

## 6) Predict test & write submission

In [ ]:
pred = best_model.predict(X_test)
save_submission(pred, SUBMISSION_PATH)
print(f"Wrote {SUBMISSION_PATH}")

---

### Notes
- This notebook keeps EDA light on purpose; deeper error analysis lives in **03_error_analysis.ipynb**.
- After this, run **02_cnn_model.ipynb** to train a stronger CNN and overwrite `submission_baseline.csv` with the CNN version (`submission_cnn_v1.csv`).